# Exploración y Entrenamiento AutoML de Longevidad
Este notebook ejecuta la integración final de la capa Gold y el entrenamiento predictivo con TPOT (algoritmos genéticos).

In [1]:
import pandas as pd
from tpot import TPOTRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')

print('Librerías cargadas correctamente.')

Librerías cargadas correctamente.


## 1. Extracción e Integración de Capas Gold
Cruzamos las bases de datos procesadas por los Miembros 1, 2 y 3 utilizando el identificador SEQN.

In [2]:
# Carga de archivos Parquet generados por Kedro
m1 = pd.read_parquet('../data/03_primary/member1_gold.parquet')
m2 = pd.read_parquet('../data/03_primary/member2_gold.parquet')
m3 = pd.read_parquet('../data/03_primary/member3_gold.parquet')

# Asegurar tipo de dato para el cruce
m1['SEQN'] = m1['SEQN'].astype(int)
m2['SEQN'] = m2['SEQN'].astype(int)
m3['SEQN'] = m3['SEQN'].astype(int)

# Cruce a 3 bandas
df = m1.merge(m2, on='SEQN', how='inner').merge(m3, on='SEQN', how='inner')

print(f'Master Table consolidada con {len(df)} pacientes.')
df.head(3)

Master Table consolidada con 4598 pacientes.


,SEQN,SDDSRVYR_x,RIDSTATR_x,RIAGENDR_x,RIDAGEYR_x,RIDAGEMN_x,RIDRETH1_x,RIDRETH3_x,RIDEXMON_x,RIDEXAGM_x,...,renal_inflammatory_score,longevity_risk_index,risk_tier,MORTSTAT,PERMTH_EXM,DIABETES,HYPERTEN,pipeline_member,pipeline_version,data_sections
0,93709,10.0,2.0,2.0,75.0,NaN,4.0,4.0,1.0,NaN,...,0.5,20.0,Moderate,0.0,6.0,NaN,NaN,member3_matias_retamal,2.0.0,laboratory|limited_access
1,93711,10.0,2.0,1.0,56.0,NaN,5.0,6.0,2.0,NaN,...,0.0,0.0,Low,0.0,2.0,NaN,NaN,member3_matias_retamal,2.0.0,laboratory|limited_access
2,93713,10.0,2.0,1.0,67.0,NaN,3.0,3.0,1.0,NaN,...,0.0,0.0,Low,0.0,4.0,NaN,NaN,member3_matias_retamal,2.0.0,laboratory|limited_access


## 2. Feature Engineering (Ingeniería de Características)
Calculamos las variables predictoras (X) y el objetivo (y) usando la lógica de negocio clínica.

In [3]:
# Demografía
age_col = [c for c in df.columns if 'RIDAGEYR' in c or 'age_years' in c][0]
gender_col = [c for c in df.columns if 'RIAGENDR' in c or 'gender' in c][0]

df['age_years'] = df[age_col]
if df[gender_col].dtype == object:
    df['gender_encoded'] = df[gender_col].apply(lambda x: 1 if x in ['Hombre', 'Masculino'] else 0)
else:
    df['gender_encoded'] = df[gender_col].apply(lambda x: 1 if x == 1.0 else 0)

# Nutrición
def calc_nutri_quality(row):
    score = 100
    kcal = row.get('AVG_KCAL', 2000)
    if pd.isna(kcal): kcal = 2000
    if kcal > 3500 or kcal < 800: score -= 40
    elif kcal > 2800 or kcal < 1200: score -= 20
    
    prot = row.get('AVG_PROT', 60)
    if pd.isna(prot): prot = 60
    if prot < 40: score -= 30
    return max(0, score)

df['nutritional_quality_score'] = df.apply(calc_nutri_quality, axis=1)

# Biomarcadores
df['bmi'] = df['BMXBMI'].fillna(df['BMXBMI'].median())
df['glucose'] = df['LBXSGL'].fillna(df['LBXSGL'].median())

# Filtrar target nulo
df = df.dropna(subset=['healthy_aging_score'])

print('Feature Engineering completado.')

Feature Engineering completado.


## 3. Entrenamiento con TPOT AutoML
Entrenamos el modelo iterando sobre arquitecturas para maximizar el R2.

In [4]:
X = df[['age_years', 'gender_encoded', 'nutritional_quality_score', 'bmi', 'glucose']]
y = df['healthy_aging_score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Instanciamos TPOT (limitado a 2 minutos para efectos de demostración)
tpot = TPOTRegressor(
    generations=1,
    population_size=3,
    cv=2,
    verbose=2,
    random_state=42,
    n_jobs=1
)

# Ejecutamos el entrenamiento evolutivo
tpot.fit(X_train, y_train)

print(f'Score R2 en datos de prueba: {tpot.score(X_test, y_test):.4f}')

Generation: : 1it [06:50, 410.79s/it]


AttributeError: 'TPOTRegressor' object has no attribute 'score'

## 4. Exportación del Modelo
Exportamos el pipeline de scikit-learn ganador para ser inyectado en FastAPI.

In [ ]:
# Exportar el pipeline óptimo encontrado por TPOT
tpot.export('tpot_optimal_pipeline.py')

print('El pipeline ha sido exportado a tpot_optimal_pipeline.py')